In [1]:
from pathlib import Path
import importlib.metadata as metadata

# Locate the uploaded code.
candidates = [
    Path("/kaggle/input/ich-repaired-code"),
    Path("/kaggle/input/datasets/mabdulal/ich-repaired-code"),
]

code_folder = next((p for p in candidates if p.exists()), None)

if code_folder:
    print("CODE LOCATION:", code_folder)
    print("\nUploaded files:")
    for path in sorted(code_folder.rglob("*")):
        if path.is_file():
            print(path.relative_to(code_folder))
else:
    print("Please expand ich-repaired-code in the Input panel")
    print("and copy its folder path.")

# Check installed dependencies without changing the environment.
print("\nInstalled versions:")
for package in [
    "tensorflow", "keras", "numpy", "pandas",
    "pydicom", "scikit-learn", "scikit-image", "pytest"
]:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

CODE LOCATION: /kaggle/input/datasets/mabdulal/ich-repaired-code

Uploaded files:
ich/__init__.py
ich/cli.py
ich/dataset.py
ich/evaluation.py
ich/explainability.py
ich/losses.py
ich/metrics.py
ich/models.py
ich/preprocessing.py
ich/training.py
ich/transforms.py
pyproject.toml
tests/conftest.py
tests/test_integrity.py
tests/test_models.py
tests/test_pipeline.py

Installed versions:
tensorflow: 2.20.0
keras: 3.13.2
numpy: 2.0.2
pandas: 2.3.3
pydicom: 3.0.2
scikit-learn: 1.6.1
scikit-image: 0.25.2
pytest: 8.4.2


In [4]:
from pathlib import Path
import shutil
import os

source = Path("/kaggle/input/datasets/mabdulal/ich-repaired-code")
work = Path("/kaggle/working/ich-research")

work.mkdir(parents=True, exist_ok=True)

for folder in ["ich", "tests"]:
    shutil.copytree(source / folder, work / folder, dirs_exist_ok=True)

shutil.copy2(source / "pyproject.toml", work / "pyproject.toml")

os.chdir(work)

assert (work / "ich" / "__init__.py").exists(), "Missing __init__.py"
print("Ready:", work)

Ready: /kaggle/working/ich-research


In [5]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q",
     "--junitxml=/kaggle/working/test-results.xml"],
    cwd="/kaggle/working/ich-research",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print(result.stdout)
print("Test exit code:", result.returncode)

................................                                         [100%]
=============================== warnings summary ===============================
../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64
  /usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
    prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))

../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85
../../../usr/local/lib/python3.12/dist-packages/matplotlib/_fontconfig_pattern.py:85
  /usr/local/l

In [6]:
from pathlib import Path
import sys
import pandas as pd
import pydicom

sys.path.insert(0, "/kaggle/working/ich-research")
from ich.preprocessing import dicom_to_hu, window_image

# Locate the RSNA training dataset.
name = "rsna-intracranial-hemorrhage-detection"
roots = [
    Path("/kaggle/input") / name,
    Path("/kaggle/input/competitions") / name,
]
candidates = roots + [root / name for root in roots]

data_dir = next(
    (p for p in candidates
     if (p / "stage_2_train.csv").exists()
     and (p / "stage_2_train").is_dir()),
    None,
)

if data_dir is None:
    raise FileNotFoundError(
        "Expand the RSNA dataset in Input and send its folder path."
    )

print("Dataset location:", data_dir)

# Read a small number of label rows and check five distinct images.
rows = pd.read_csv(data_dir / "stage_2_train.csv", nrows=120)
image_ids = rows["ID"].str.rsplit("_", n=1).str[0].unique()[:5]

required = [
    "PatientID", "StudyInstanceUID",
    "SeriesInstanceUID", "SOPInstanceUID"
]

for number, image_id in enumerate(image_ids, start=1):
    ds = pydicom.dcmread(
        data_dir / "stage_2_train" / f"{image_id}.dcm"
    )
    missing = [
        tag for tag in required
        if not str(getattr(ds, tag, "")).strip()
    ]
    if missing:
        raise ValueError(f"Image {number}: missing {missing}")

    hu = dicom_to_hu(ds)
    image = window_image(hu)

    print(
        f"Image {number}: OK | "
        f"HU range {hu.min():.1f} to {hu.max():.1f} | "
        f"processed shape {image.shape}"
    )

print("Small real-DICOM check completed.")

Dataset location: /kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection


/usr/local/lib/python3.12/dist-packages/pydicom/valuerep.py:440: UserWarning: Invalid value for VR UI: 'ID_6dec708c74'. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#table_6.2-1> for allowed values for each VR.
  warn_and_log(msg)
/usr/local/lib/python3.12/dist-packages/pydicom/valuerep.py:440: UserWarning: Invalid value for VR UI: 'ID_1b17a4a944'. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#table_6.2-1> for allowed values for each VR.
  warn_and_log(msg)
/usr/local/lib/python3.12/dist-packages/pydicom/valuerep.py:440: UserWarning: Invalid value for VR UI: 'ID_12cadc6af'. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#table_6.2-1> for allowed values for each VR.
  warn_and_log(msg)


Image 1: OK | HU range -3024.0 to 1742.0 | processed shape (224, 224, 3)
Image 2: OK | HU range -3024.0 to 362.0 | processed shape (224, 224, 3)
Image 3: OK | HU range -1018.0 to 1496.0 | processed shape (224, 224, 3)
Image 4: OK | HU range -3024.0 to 3071.0 | processed shape (224, 224, 3)


/usr/local/lib/python3.12/dist-packages/pydicom/valuerep.py:440: UserWarning: Invalid value for VR UI: 'ID_0a9ac70962'. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#table_6.2-1> for allowed values for each VR.
  warn_and_log(msg)
/usr/local/lib/python3.12/dist-packages/pydicom/valuerep.py:440: UserWarning: Invalid value for VR UI: 'ID_3f92acee54'. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#table_6.2-1> for allowed values for each VR.
  warn_and_log(msg)
/usr/local/lib/python3.12/dist-packages/pydicom/valuerep.py:440: UserWarning: Invalid value for VR UI: 'ID_38fd7baa0'. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#table_6.2-1> for allowed values for each VR.
  warn_and_log(msg)
/usr/local/lib/python3.12/dist-packages/pydicom/valuerep.py:440: UserWarning: Invalid value for VR UI: 'ID_45e4c06199'. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#ta

Image 5: OK | HU range -1024.0 to 1059.0 | processed shape (224, 224, 3)
Small real-DICOM check completed.


In [7]:
import numpy as np
import pandas as pd
import pydicom

records = []

for number, image_id in enumerate(image_ids, start=1):
    ds = pydicom.dcmread(
        data_dir / "stage_2_train" / f"{image_id}.dcm"
    )
    pixels = ds.pixel_array
    hu = dicom_to_hu(ds)

    records.append({
        "image": number,
        "pixel_dtype": str(pixels.dtype),
        "signed_pixels": int(ds.PixelRepresentation),
        "bits_stored": int(ds.BitsStored),
        "slope": float(ds.RescaleSlope),
        "intercept": float(ds.RescaleIntercept),
        "padding_value": str(
            getattr(ds, "PixelPaddingValue", "absent")
        ),
        "padding_limit": str(
            getattr(ds, "PixelPaddingRangeLimit", "absent")
        ),
        "raw_min": int(pixels.min()),
        "raw_max": int(pixels.max()),
        "HU_min": float(hu.min()),
        "HU_max": float(hu.max()),
        "percent_below_minus_1100": round(
            float(np.mean(hu < -1100) * 100), 2
        ),
    })

print(pd.DataFrame(records).to_string(index=False))

 image pixel_dtype  signed_pixels  bits_stored  slope  intercept padding_value padding_limit  raw_min  raw_max  HU_min  HU_max  percent_below_minus_1100
     1       int16              1           16    1.0    -1024.0        absent        absent    -2000     2766 -3024.0  1742.0                     21.28
     2       int16              1           16    1.0    -1024.0        absent        absent    -2000     1386 -3024.0   362.0                     21.28
     3      uint16              0           12    1.0    -1024.0        absent        absent        6     2520 -1018.0  1496.0                      0.00
     4       int16              1           16    1.0    -1024.0        absent        absent    -2000     4095 -3024.0  3071.0                     21.28
     5      uint16              0           12    1.0    -1024.0        absent        absent        0     2083 -1024.0  1059.0                      0.00


In [8]:
from pathlib import Path
import subprocess
import sys
import os

project = Path("/kaggle/working/ich-research")
manifest = Path("/kaggle/working/rsna-manifest.csv")
log_path = Path("/kaggle/working/prepare-manifest.log")

command = [
    sys.executable, "-u", "-m", "ich.cli", "prepare",
    "--labels", str(data_dir / "stage_2_train.csv"),
    "--dicoms", str(data_dir / "stage_2_train"),
    "--output", str(manifest),
    "--seed", "42",
]

print("Preparing the manifest. Keep this cell running.", flush=True)
print("Detailed warnings and errors are saved to:", log_path, flush=True)

# Write output to disk rather than flooding the notebook with UID warnings.
with log_path.open("w") as log:
    result = subprocess.run(
        command,
        cwd=project,
        stdout=log,
        stderr=subprocess.STDOUT,
    )

print("Exit code:", result.returncode)

# Show only the end of the log.
with log_path.open("rb") as log:
    log.seek(max(0, os.path.getsize(log_path) - 6000))
    print(log.read().decode("utf-8", errors="replace"))

if result.returncode == 0:
    print("Manifest created:", manifest)
else:
    print("Preparation stopped. Send the output above before proceeding.")

Preparing the manifest. Keep this cell running.
Detailed warnings and errors are saved to: /kaggle/working/prepare-manifest.log
Exit code: 1
59'. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#table_6.2-1> for allowed values for each VR.
  warn_and_log(msg)
/usr/local/lib/python3.12/dist-packages/pydicom/valuerep.py:440: UserWarning: Invalid value for VR UI: 'ID_6430404fe'. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#table_6.2-1> for allowed values for each VR.
  warn_and_log(msg)
/usr/local/lib/python3.12/dist-packages/pydicom/valuerep.py:440: UserWarning: Invalid value for VR UI: 'ID_64305bfd1'. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#table_6.2-1> for allowed values for each VR.
  warn_and_log(msg)
/usr/local/lib/python3.12/dist-packages/pydicom/valuerep.py:440: UserWarning: Invalid value for VR UI: 'ID_64308a64e'. Please see <https://dicom.nema.org/medical/dicom/current/ou

In [10]:
from pathlib import Path
import pydicom

folder = Path(
    "/kaggle/input/competitions/"
    "rsna-intracranial-hemorrhage-detection/"
    "rsna-intracranial-hemorrhage-detection/stage_2_train"
)
path = folder / "ID_6431af929.dcm"

if not path.exists():
    print("Candidate file not found; its identity is not confirmed.")
else:
    print("Checking:", path.name)
    print("File size:", path.stat().st_size, "bytes")

    ds = pydicom.dcmread(path)

    for tag in [
        "Rows", "Columns", "NumberOfFrames",
        "SamplesPerPixel", "BitsAllocated", "BitsStored",
        "PixelRepresentation", "PhotometricInterpretation",
        "RescaleSlope", "RescaleIntercept"
    ]:
        print(f"{tag}: {getattr(ds, tag, 'absent')}")

    print(
        "TransferSyntaxUID:",
        getattr(ds.file_meta, "TransferSyntaxUID", "absent")
    )
    print("PixelData bytes:", len(ds.get("PixelData", b"")))

    try:
        pixels = ds.pixel_array
        print("Decoding succeeded:", pixels.shape, pixels.dtype)
        print("This candidate did not reproduce the failure.")
    except Exception as error:
        print("DECODING FAILED:", type(error).__name__, str(error))

Checking: ID_6431af929.dcm
File size: 154410 bytes
Rows: 512
Columns: 512
NumberOfFrames: absent
SamplesPerPixel: 1
BitsAllocated: 16
BitsStored: 16
PixelRepresentation: 1
PhotometricInterpretation: MONOCHROME2
RescaleSlope: 1
RescaleIntercept: -1024
TransferSyntaxUID: 1.2.840.10008.1.2.1
PixelData bytes: 153710
DECODING FAILED: ValueError The number of bytes of pixel data is less than expected (153710 vs 524288 bytes) - the dataset may be corrupted, have an invalid group 0028 element value, or the transfer syntax may be incorrect


In [11]:
# Paste this entire file into ONE new Kaggle code cell and run it.
from pathlib import Path
import shutil
import sys

project = Path("/kaggle/working/ich-research")
source = Path("/kaggle/input/datasets/mabdulal/ich-repaired-code")
if not (project / "ich" / "__init__.py").exists():
    shutil.copytree(source, project, dirs_exist_ok=True)
sys.path.insert(0, str(project))

"""Checkpointed preparation for immutable datasets, with explicit exclusion evidence."""
import hashlib
import json
from pathlib import Path
import sqlite3
import time
import warnings
import numpy as np
import pandas as pd
import pydicom
from ich.dataset import read_labels, split_manifest
from ich import preprocessing


def file_hash(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def scan_one(image_id, path):
    """All failures retain their image ID; only the known payload defect is eligible."""
    record = {"image_id": image_id, "path": str(path), "status": "error", "uid_warnings": 0}
    captured = []
    try:
        with warnings.catch_warnings(record=True) as captured:
            warnings.simplefilter("always", UserWarning)
            ds = pydicom.dcmread(path)
            for key, tag in zip(("patient_id", "study_id", "series_id", "sop_id"),
                                ("PatientID", "StudyInstanceUID", "SeriesInstanceUID", "SOPInstanceUID")):
                record[key] = str(getattr(ds, tag, "")).strip()
                if not record[key]:
                    raise ValueError(f"Missing identity: {tag}")
            payload = len(ds.get("PixelData", b""))
            record["payload_bytes"] = payload
            # An exact fingerprint of the defect observed on Kaggle, not a general skip rule.
            signature = (path.stat().st_size, int(ds.Rows), int(ds.Columns),
                int(getattr(ds, "NumberOfFrames", 1)), int(ds.SamplesPerPixel), int(ds.BitsAllocated),
                int(ds.BitsStored), int(ds.PixelRepresentation), str(ds.file_meta.TransferSyntaxUID), payload)
            known = image_id == "ID_6431af929" and signature == (
                154410, 512, 512, 1, 1, 16, 16, 1, "1.2.840.10008.1.2.1", 153710)
            try:
                hu = np.ascontiguousarray(preprocessing.dicom_to_hu(ds), dtype="<f4")
            except ValueError as error:
                if known and "number of bytes of pixel data is less than expected" in str(error):
                    record.update(status="known_short_payload", error=str(error),
                                  file_sha256=file_hash(path), expected_payload_bytes=524288)
                    return record
                raise
            record.update(status="ok", pixel_hash=hashlib.sha256(str(hu.shape).encode()+hu.tobytes()).hexdigest())
    except Exception as error:
        record["error"] = f"{type(error).__name__}: {error}"
    finally:
        record["uid_warnings"] = sum("Invalid value for VR UI:" in str(w.message) for w in captured)
        other = [str(w.message) for w in captured if "Invalid value for VR UI:" not in str(w.message)]
        if other:
            record["other_warnings"] = other
    return record


def prepare_resumable(labels_csv, dicom_dir, output, checkpoint_dir, seed=42,
                      exclude_confirmed_short_payload=False, progress_every=1000):
    """Resume unchanged, immutable input mounts. Unexpected errors block final splitting.

    Preserve checkpoint_dir between sessions. Source location, labels, reader code,
    pydicom/numpy versions and per-file size/mtime are bound to the cache. For mutable
    datasets use a new checkpoint directory; size/mtime is not a content proof.
    """
    output, checkpoint_dir = Path(output), Path(checkpoint_dir)
    dicom_dir = Path(dicom_dir).resolve()
    if output.exists():
        raise FileExistsError(f"Existing manifest protected: {output}")
    if progress_every < 1:
        raise ValueError("progress_every must be positive")
    labels = read_labels(labels_csv)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    contract = dict(schema=1, dicom_dir=str(dicom_dir), labels_sha256=file_hash(labels_csv),
                    preprocessing_sha256=file_hash(preprocessing.__file__),
                    scanner_version="short-payload-review-v1",
                    pydicom=pydicom.__version__, numpy=np.__version__)
    contract_json = json.dumps(contract, sort_keys=True)
    db = sqlite3.connect(checkpoint_dir/"scan.sqlite")
    started = time.monotonic()
    scanned = cached = 0
    records = []
    print(f"Checking {len(labels):,} images. Checkpoint: {checkpoint_dir}", flush=True)
    try:
        db.execute("CREATE TABLE IF NOT EXISTS config (value TEXT NOT NULL)")
        db.execute("CREATE TABLE IF NOT EXISTS images (id TEXT PRIMARY KEY, signature TEXT, record TEXT)")
        previous = db.execute("SELECT value FROM config").fetchone()
        if previous and previous[0] != contract_json:
            raise ValueError("Checkpoint source/reader contract changed; use a new checkpoint directory")
        if not previous:
            db.execute("INSERT INTO config VALUES (?)", (contract_json,))
            db.commit()
        for index, image_id in enumerate(labels.image_id, start=1):
            path = dicom_dir/f"{image_id}.dcm"
            stat = path.stat() if path.exists() else None
            signature = json.dumps([stat.st_size, stat.st_mtime_ns] if stat else None)
            previous = db.execute("SELECT signature, record FROM images WHERE id=?", (image_id,)).fetchone()
            if previous and previous[0] == signature:
                record = json.loads(previous[1])
                cached += 1
            else:
                record = scan_one(image_id, path)
                db.execute("INSERT OR REPLACE INTO images VALUES (?, ?, ?)", (image_id, signature, json.dumps(record)))
                scanned += 1
            records.append(record)
            if index % 100 == 0:
                db.commit()
            if index % progress_every == 0 or index == len(labels):
                elapsed = (time.monotonic()-started)/60
                print(f"{index:,}/{len(labels):,} checked | {cached:,} cached | {scanned:,} read | {elapsed:.1f} min", flush=True)
    finally:
        db.commit()
        db.close()
    accepted = [r for r in records if r["status"] == "known_short_payload" and exclude_confirmed_short_payload]
    errors = [r for r in records if r["status"] != "ok" and r not in accepted]
    # Always write inspection evidence before attempting the final split.
    for name, rows in (("exclusions.json", accepted), ("unresolved-errors.json", errors)):
        (checkpoint_dir/name).write_text(json.dumps(rows, indent=2), encoding="utf-8")
    warning_records = [{"image_id": r["image_id"], "uid_warnings": r["uid_warnings"],
                        "other_warnings": r.get("other_warnings", [])}
                       for r in records if r["uid_warnings"] or r.get("other_warnings")]
    with (checkpoint_dir/"warnings.jsonl").open("w", encoding="utf-8") as stream:
        for record in warning_records:
            stream.write(json.dumps(record)+"\n")
    summary = dict(total_images=len(labels), readable=sum(r["status"] == "ok" for r in records),
                   excluded=len(accepted), unresolved=len(errors), seed=seed,
                   uid_warning_count=sum(r["uid_warnings"] for r in records), contract=contract)
    (checkpoint_dir/"summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    if errors:
        print(json.dumps(errors[:10], indent=2), flush=True)
        raise ValueError(f"{len(errors)} unresolved image errors; see unresolved-errors.json. No manifest created.")
    kept = pd.DataFrame([{key: r[key] for key in (
        "image_id", "path", "patient_id", "study_id", "series_id", "sop_id", "pixel_hash")}
        for r in records if r["status"] == "ok"])
    frame = kept.merge(labels, on="image_id", validate="one_to_one")
    excluded_labels = labels[labels.image_id.isin([r["image_id"] for r in accepted])]
    excluded_labels.to_csv(checkpoint_dir/"excluded-labels.csv", index=False)
    frame = split_manifest(frame, seed=seed)  # All existing identity/leakage checks remain mandatory.
    output.parent.mkdir(parents=True, exist_ok=True)
    temporary = output.with_suffix(".csv.tmp")
    frame.to_csv(temporary, index=False)
    temporary.replace(output)
    print(frame.groupby("partition").agg(images=("image_id", "size"), patients=("patient_id", "nunique")), flush=True)
    print(f"Saved {output}; documented exclusions: {len(accepted)}", flush=True)
    return frame


# The only accepted image exclusion is the exact short-payload failure
# reproduced for ID_6431af929. Other failures block the final manifest.
data = Path("/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection")
manifest_path = Path("/kaggle/working/rsna-manifest.csv")
checkpoint_path = Path("/kaggle/working/rsna-manifest-checkpoint")

if manifest_path.exists():
    print("An existing manifest is protected:", manifest_path)
else:
    prepare_resumable(
        data / "stage_2_train.csv",
        data / "stage_2_train",
        manifest_path,
        checkpoint_path,
        seed=42,
        exclude_confirmed_short_payload=True,
        progress_every=1000,
    )


Checking 752,803 images. Checkpoint: /kaggle/working/rsna-manifest-checkpoint
1,000/752,803 checked | 0 cached | 1,000 read | 0.5 min
2,000/752,803 checked | 0 cached | 2,000 read | 1.1 min
3,000/752,803 checked | 0 cached | 3,000 read | 1.6 min
4,000/752,803 checked | 0 cached | 4,000 read | 2.1 min
5,000/752,803 checked | 0 cached | 5,000 read | 2.7 min
6,000/752,803 checked | 0 cached | 6,000 read | 3.3 min
7,000/752,803 checked | 0 cached | 7,000 read | 3.8 min
8,000/752,803 checked | 0 cached | 8,000 read | 4.3 min
9,000/752,803 checked | 0 cached | 9,000 read | 4.9 min
10,000/752,803 checked | 0 cached | 10,000 read | 5.4 min
11,000/752,803 checked | 0 cached | 11,000 read | 5.9 min
12,000/752,803 checked | 0 cached | 12,000 read | 6.4 min
13,000/752,803 checked | 0 cached | 13,000 read | 6.9 min
14,000/752,803 checked | 0 cached | 14,000 read | 7.3 min
15,000/752,803 checked | 0 cached | 15,000 read | 7.8 min
16,000/752,803 checked | 0 cached | 16,000 read | 8.2 min
17,000/752,8

KeyboardInterrupt: 

In [12]:
from pathlib import Path
import shutil
import os
from IPython.display import FileLink, display

checkpoint = Path("/kaggle/working/rsna-manifest-checkpoint")
assert (checkpoint / "scan.sqlite").exists(), "Checkpoint not found"

shutil.make_archive(
    "/kaggle/working/rsna-checkpoint-backup",
    "zip",
    root_dir=checkpoint.parent,
    base_dir=checkpoint.name,
)

os.chdir("/kaggle/working")
display(FileLink("rsna-checkpoint-backup.zip"))

/kaggle/working/rsna-checkpoint-backup.zip

In [1]:
from pathlib import Path
import sqlite3

checkpoint = Path(
    "/kaggle/working/rsna-manifest-checkpoint/scan.sqlite"
)

if checkpoint.exists():
    with sqlite3.connect(
        checkpoint.as_uri() + "?mode=ro", uri=True
    ) as connection:
        count = connection.execute(
            "SELECT COUNT(*) FROM images"
        ).fetchone()[0]

    print(f"Checkpoint found: {count:,} image records saved.")
else:
    print("Checkpoint missing. Restore rsna-checkpoint-backup.zip first.")

Checkpoint missing. Restore rsna-checkpoint-backup.zip first.


In [1]:
from pathlib import Path
import shutil
import sqlite3

# Locate the uploaded checkpoint dataset.
roots = [
    Path("/kaggle/input/datasets/mabdulal/checkpoint-backup"),
    Path("/kaggle/input/checkpoint-backup"),
    Path("/kaggle/input/datasets/mabdulal/ich-checkpoint-backup"),
    Path("/kaggle/input/ich-checkpoint-backup"),
]

matches = [
    file
    for root in roots if root.exists()
    for file in root.rglob("scan.sqlite")
]

assert len(matches) == 1, (
    "Could not identify one checkpoint database. "
    "Expand Checkpoint-backup in Input and send a screenshot."
)

source = matches[0]
destination = Path(
    "/kaggle/working/rsna-manifest-checkpoint/scan.sqlite"
)

def check_database(path):
    with sqlite3.connect(path.as_uri() + "?mode=ro", uri=True) as db:
        assert db.execute("PRAGMA quick_check").fetchone()[0] == "ok"
        return db.execute("SELECT COUNT(*) FROM images").fetchone()[0]

# Verify the backup before restoring it.
print(f"Backup contains {check_database(source):,} records.")

if destination.exists():
    print("Keeping the existing working checkpoint to protect progress.")
else:
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    print("Checkpoint restored.")

print(f"Ready to resume: {check_database(destination):,} saved records.")

Backup contains 478,050 records.
Checkpoint restored.
Ready to resume: 478,050 saved records.


In [ ]:
# Paste this entire file into ONE new Kaggle code cell and run it.
from pathlib import Path
import shutil
import sys

project = Path("/kaggle/working/ich-research")
source = Path("/kaggle/input/datasets/mabdulal/ich-repaired-code")
if not (project / "ich" / "__init__.py").exists():
    shutil.copytree(source, project, dirs_exist_ok=True)
sys.path.insert(0, str(project))

"""Checkpointed preparation for immutable datasets, with explicit exclusion evidence."""
import hashlib
import json
from pathlib import Path
import sqlite3
import time
import warnings
import numpy as np
import pandas as pd
import pydicom
from ich.dataset import read_labels, split_manifest
from ich import preprocessing


def file_hash(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def scan_one(image_id, path):
    """All failures retain their image ID; only the known payload defect is eligible."""
    record = {"image_id": image_id, "path": str(path), "status": "error", "uid_warnings": 0}
    captured = []
    try:
        with warnings.catch_warnings(record=True) as captured:
            warnings.simplefilter("always", UserWarning)
            ds = pydicom.dcmread(path)
            for key, tag in zip(("patient_id", "study_id", "series_id", "sop_id"),
                                ("PatientID", "StudyInstanceUID", "SeriesInstanceUID", "SOPInstanceUID")):
                record[key] = str(getattr(ds, tag, "")).strip()
                if not record[key]:
                    raise ValueError(f"Missing identity: {tag}")
            payload = len(ds.get("PixelData", b""))
            record["payload_bytes"] = payload
            # An exact fingerprint of the defect observed on Kaggle, not a general skip rule.
            signature = (path.stat().st_size, int(ds.Rows), int(ds.Columns),
                int(getattr(ds, "NumberOfFrames", 1)), int(ds.SamplesPerPixel), int(ds.BitsAllocated),
                int(ds.BitsStored), int(ds.PixelRepresentation), str(ds.file_meta.TransferSyntaxUID), payload)
            known = image_id == "ID_6431af929" and signature == (
                154410, 512, 512, 1, 1, 16, 16, 1, "1.2.840.10008.1.2.1", 153710)
            try:
                hu = np.ascontiguousarray(preprocessing.dicom_to_hu(ds), dtype="<f4")
            except ValueError as error:
                if known and "number of bytes of pixel data is less than expected" in str(error):
                    record.update(status="known_short_payload", error=str(error),
                                  file_sha256=file_hash(path), expected_payload_bytes=524288)
                    return record
                raise
            record.update(status="ok", pixel_hash=hashlib.sha256(str(hu.shape).encode()+hu.tobytes()).hexdigest())
    except Exception as error:
        record["error"] = f"{type(error).__name__}: {error}"
    finally:
        record["uid_warnings"] = sum("Invalid value for VR UI:" in str(w.message) for w in captured)
        other = [str(w.message) for w in captured if "Invalid value for VR UI:" not in str(w.message)]
        if other:
            record["other_warnings"] = other
    return record


def prepare_resumable(labels_csv, dicom_dir, output, checkpoint_dir, seed=42,
                      exclude_confirmed_short_payload=False, progress_every=1000):
    """Resume unchanged, immutable input mounts. Unexpected errors block final splitting.

    Preserve checkpoint_dir between sessions. Source location, labels, reader code,
    pydicom/numpy versions and per-file size/mtime are bound to the cache. For mutable
    datasets use a new checkpoint directory; size/mtime is not a content proof.
    """
    output, checkpoint_dir = Path(output), Path(checkpoint_dir)
    dicom_dir = Path(dicom_dir).resolve()
    if output.exists():
        raise FileExistsError(f"Existing manifest protected: {output}")
    if progress_every < 1:
        raise ValueError("progress_every must be positive")
    labels = read_labels(labels_csv)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    contract = dict(schema=1, dicom_dir=str(dicom_dir), labels_sha256=file_hash(labels_csv),
                    preprocessing_sha256=file_hash(preprocessing.__file__),
                    scanner_version="short-payload-review-v1",
                    pydicom=pydicom.__version__, numpy=np.__version__)
    contract_json = json.dumps(contract, sort_keys=True)
    db = sqlite3.connect(checkpoint_dir/"scan.sqlite")
    started = time.monotonic()
    scanned = cached = 0
    records = []
    print(f"Checking {len(labels):,} images. Checkpoint: {checkpoint_dir}", flush=True)
    try:
        db.execute("CREATE TABLE IF NOT EXISTS config (value TEXT NOT NULL)")
        db.execute("CREATE TABLE IF NOT EXISTS images (id TEXT PRIMARY KEY, signature TEXT, record TEXT)")
        previous = db.execute("SELECT value FROM config").fetchone()
        if previous and previous[0] != contract_json:
            raise ValueError("Checkpoint source/reader contract changed; use a new checkpoint directory")
        if not previous:
            db.execute("INSERT INTO config VALUES (?)", (contract_json,))
            db.commit()
        for index, image_id in enumerate(labels.image_id, start=1):
            path = dicom_dir/f"{image_id}.dcm"
            stat = path.stat() if path.exists() else None
            signature = json.dumps([stat.st_size, stat.st_mtime_ns] if stat else None)
            previous = db.execute("SELECT signature, record FROM images WHERE id=?", (image_id,)).fetchone()
            if previous and previous[0] == signature:
                record = json.loads(previous[1])
                cached += 1
            else:
                record = scan_one(image_id, path)
                db.execute("INSERT OR REPLACE INTO images VALUES (?, ?, ?)", (image_id, signature, json.dumps(record)))
                scanned += 1
            records.append(record)
            if index % 100 == 0:
                db.commit()
            if index % progress_every == 0 or index == len(labels):
                elapsed = (time.monotonic()-started)/60
                print(f"{index:,}/{len(labels):,} checked | {cached:,} cached | {scanned:,} read | {elapsed:.1f} min", flush=True)
    finally:
        db.commit()
        db.close()
    accepted = [r for r in records if r["status"] == "known_short_payload" and exclude_confirmed_short_payload]
    errors = [r for r in records if r["status"] != "ok" and r not in accepted]
    # Always write inspection evidence before attempting the final split.
    for name, rows in (("exclusions.json", accepted), ("unresolved-errors.json", errors)):
        (checkpoint_dir/name).write_text(json.dumps(rows, indent=2), encoding="utf-8")
    warning_records = [{"image_id": r["image_id"], "uid_warnings": r["uid_warnings"],
                        "other_warnings": r.get("other_warnings", [])}
                       for r in records if r["uid_warnings"] or r.get("other_warnings")]
    with (checkpoint_dir/"warnings.jsonl").open("w", encoding="utf-8") as stream:
        for record in warning_records:
            stream.write(json.dumps(record)+"\n")
    summary = dict(total_images=len(labels), readable=sum(r["status"] == "ok" for r in records),
                   excluded=len(accepted), unresolved=len(errors), seed=seed,
                   uid_warning_count=sum(r["uid_warnings"] for r in records), contract=contract)
    (checkpoint_dir/"summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    if errors:
        print(json.dumps(errors[:10], indent=2), flush=True)
        raise ValueError(f"{len(errors)} unresolved image errors; see unresolved-errors.json. No manifest created.")
    kept = pd.DataFrame([{key: r[key] for key in (
        "image_id", "path", "patient_id", "study_id", "series_id", "sop_id", "pixel_hash")}
        for r in records if r["status"] == "ok"])
    frame = kept.merge(labels, on="image_id", validate="one_to_one")
    excluded_labels = labels[labels.image_id.isin([r["image_id"] for r in accepted])]
    excluded_labels.to_csv(checkpoint_dir/"excluded-labels.csv", index=False)
    frame = split_manifest(frame, seed=seed)  # All existing identity/leakage checks remain mandatory.
    output.parent.mkdir(parents=True, exist_ok=True)
    temporary = output.with_suffix(".csv.tmp")
    frame.to_csv(temporary, index=False)
    temporary.replace(output)
    print(frame.groupby("partition").agg(images=("image_id", "size"), patients=("patient_id", "nunique")), flush=True)
    print(f"Saved {output}; documented exclusions: {len(accepted)}", flush=True)
    return frame


# The only accepted image exclusion is the exact short-payload failure
# reproduced for ID_6431af929. Other failures block the final manifest.
data = Path("/kaggle/input/competitions/rsna-intracranial-hemorrhage-detection/rsna-intracranial-hemorrhage-detection")
manifest_path = Path("/kaggle/working/rsna-manifest.csv")
checkpoint_path = Path("/kaggle/working/rsna-manifest-checkpoint")

if manifest_path.exists():
    print("An existing manifest is protected:", manifest_path)
else:
    prepare_resumable(
        data / "stage_2_train.csv",
        data / "stage_2_train",
        manifest_path,
        checkpoint_path,
        seed=42,
        exclude_confirmed_short_payload=True,
        progress_every=1000,
    )

Checking 752,803 images. Checkpoint: /kaggle/working/rsna-manifest-checkpoint
1,000/752,803 checked | 1,000 cached | 0 read | 0.0 min
2,000/752,803 checked | 2,000 cached | 0 read | 0.1 min
3,000/752,803 checked | 3,000 cached | 0 read | 0.2 min
4,000/752,803 checked | 4,000 cached | 0 read | 0.3 min
5,000/752,803 checked | 5,000 cached | 0 read | 0.3 min
6,000/752,803 checked | 6,000 cached | 0 read | 0.4 min
7,000/752,803 checked | 7,000 cached | 0 read | 0.5 min
8,000/752,803 checked | 8,000 cached | 0 read | 0.6 min
9,000/752,803 checked | 9,000 cached | 0 read | 0.7 min
10,000/752,803 checked | 10,000 cached | 0 read | 0.7 min
11,000/752,803 checked | 11,000 cached | 0 read | 0.8 min
12,000/752,803 checked | 12,000 cached | 0 read | 0.9 min
13,000/752,803 checked | 13,000 cached | 0 read | 1.0 min
14,000/752,803 checked | 14,000 cached | 0 read | 1.1 min
15,000/752,803 checked | 15,000 cached | 0 read | 1.1 min
16,000/752,803 checked | 16,000 cached | 0 read | 1.2 min
17,000/752,8

In [ ]:
from pathlib import Path
import sqlite3
import tempfile
import zipfile
import os
from IPython.display import FileLink, display

source = Path("/kaggle/working/rsna-manifest-checkpoint/scan.sqlite")
archive = Path("/kaggle/working/rsna-checkpoint-latest.zip")
assert source.exists(), "Checkpoint not accessible"

# Make a consistent database snapshot before packaging it.
with tempfile.TemporaryDirectory() as temporary:
    snapshot = Path(temporary) / "scan.sqlite"
    src = sqlite3.connect(source.as_uri() + "?mode=ro", uri=True)
    dst = sqlite3.connect(snapshot)
    try:
        src.backup(dst)
        count = dst.execute("SELECT COUNT(*) FROM images").fetchone()[0]
        assert dst.execute("PRAGMA quick_check").fetchone()[0] == "ok"
    finally:
        dst.close()
        src.close()

    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(snapshot, "rsna-manifest-checkpoint/scan.sqlite")

print(f"Backup verified: {count:,} saved records")
os.chdir("/kaggle/working")
display(FileLink(archive.name))